# 📊 Oracle ↔ PostgresPro Data Comparator
Модульный скрипт для потокового сравнения больших таблиц (5M+ строк).
Поддерживает разные схемы, опциональные фильтры, авто-поиск PK и визуализацию.
**Архитектура:** Подключение → Метаданные → Keyset Pagination → Инкрементальное сравнение → Dashboard.

In [ ]:
import logging as _log  # Импортируем модуль структурированного логирования
import sys as _sys  # Импортируем системные утилиты для вывода в stdout
import numpy as _np  # Импортируем библиотеку быстрых числовых вычислений
import pandas as _pd  # Импортируем библиотеку табличных вычислений
import seaborn as _sns  # Импортируем библиотеку статистической визуализации
import matplotlib.pyplot as _plt  # Импортируем базовый движок отрисовки графиков
from typing import Dict, List, Optional, Any, Generator as _TypeHints  # Импортируем аннотации типов
from contextlib import contextmanager  # Импортируем декоратор контекстных менеджеров
from tqdm.notebook import tqdm as _tqdm  # Импортируем прогресс-бар для Jupyter
import warnings as _warn  # Импортируем модуль управления предупреждениями
_warn.filterwarnings("ignore")  # Отключаем некритичные алерты внешних библиотек для чистоты вывода
from concurrent.futures import ThreadPoolExecutor, as_completed  # Импортируем пул потоков для параллелизации
import hashlib as _hashlib  # Импортируем модуль хэш-функций для валидации больших объёмов
import time as _time  # Импортируем модуль времени для расчёта ETA
import os as _os  # Импортируем модуль работы с окружением

# ============================================
# Умный импорт драйверов БД: Anaconda → Python
# ============================================
# Пытаемся импортировать из окружения Anaconda, если не найдено — переключаемся на стандартное Python окружение
_oracledb = None
_psycopg = None

# Функция логирования предупреждений (должна быть определена до использования)
# Импорт oracledb: сначала пробуем anaconda, затем стандартное окружение
try:
    # Проверяем наличие в anaconda окружении
    _anaconda_path = [_p for _p in _sys.path if 'anaconda' in _p.lower() or 'miniconda' in _p.lower()]
    if _anaconda_path:
        # Приоритет anaconda путей
        for _ap in _anaconda_path:
            if _ap not in _sys.path[:3]:  # Добавляем в начало для приоритета
                _sys.path.insert(1, _ap)
    import oracledb as _oracledb_temp  # Импортируем современный тонкий драйвер Oracle
    _oracledb = _oracledb_temp
    del _oracledb_temp
except ImportError:
    try:
        # Переключаемся на стандартное Python окружение
        import oracledb as _oracledb  # Импортируем современный тонкий драйвер Oracle
    except ImportError:
        pass  # Logger будет настроен в следующей ячейке

# Импорт psycopg: сначала пробуем anaconda, затем стандартное окружение
try:
    # Проверяем наличие в anaconda окружении
    _anaconda_path = [_p for _p in _sys.path if 'anaconda' in _p.lower() or 'miniconda' in _p.lower()]
    if _anaconda_path:
        # Приоритет anaconda путей
        for _ap in _anaconda_path:
            if _ap not in _sys.path[:3]:  # Добавляем в начало для приоритета
                _sys.path.insert(1, _ap)
    import psycopg as _psycopg_temp  # Импортируем современный драйвер PostgreSQL v3
    _psycopg = _psycopg_temp
    del _psycopg_temp
except ImportError:
    try:
        # Переключаемся на стандартное Python окружение
        import psycopg as _psycopg  # Импортируем современный драйвер PostgreSQL v3
    except ImportError:
        pass  # Logger будет настроен в следующей ячейке

# Алиасы для функций из импортированных модулей (для использования в исправленном коде)
hashlib_md5 = _hashlib.md5
np_issubdtype = _np.issubdtype
np_number = _np.number
np_isclose = _np.isclose
np_isscalar = _np.isscalar
np_isnan = _np.isnan
np_array = _np.array
np_char_lower = _np.char.lower
np_count_nonzero = _np.count_nonzero
ThreadPoolExecutor = ThreadPoolExecutor
as_completed = as_completed
logger_info = lambda msg: _log.getLogger("db_comparator").info(msg)

In [ ]:
# Функция логирования предупреждений (теперь logger определён)
logger_warning = lambda msg: _log.getLogger("db_comparator").warning(msg)

# Настройка логирования в файл и консоль
_log_file = _os.environ.get('COMPARATOR_LOG_FILE', 'comparator.log')
_file_handler = _log.FileHandler(_log_file, mode='a', encoding='utf-8')
_file_handler.setLevel(_log.INFO)
_file_handler.setFormatter(_log.Formatter("%(asctime)s | %(levelname)-8s | %(message)s"))
_logger = _log.getLogger("db_comparator")
_logger.setLevel(_log.INFO)
_logger.addHandler(_file_handler)  # Добавляем файловый обработчик
_console_handler = _log.StreamHandler(_sys.stdout)
_console_handler.setFormatter(_log.Formatter("%(asctime)s | %(levelname)-8s | %(message)s"))
_logger.addHandler(_console_handler)
class DBCompError(Exception): pass
class ConnError(DBCompError): pass
class SchemaError(DBCompError): pass


In [ ]:
@contextmanager
def _ora_conn(cfg: Dict[str, str]):  # Фабрика безопасного соединения с Oracle
    conn = _oracledb.connect(user=cfg["user"], password=cfg["password"], dsn=cfg.get("dsn"))  # Инициализируем соединение в thin-режиме
    conn.call_timeout = 60000  # Устанавливаем таймаут выполнения запросов в 60 секунд
    try:
        yield conn  # Передаём активное соединение в контекст вызывающего кода
    except Exception as e:
        raise ConnError(f"Oracle connection failed: {e}") from e  # Оборачиваем ошибки в доменное исключение
    finally:
        conn.close()  # Гарантированно закрываем сессию при выходе из контекста

@contextmanager
def _pg_conn(cfg: Dict[str, str]):  # Фабрика безопасного соединения с PostgresPro
    conn = _psycopg.connect(host=cfg["host"], port=cfg["port"], dbname=cfg["dbname"], user=cfg["user"], password=cfg["password"])  # Открываем TCP-соединение v3
    conn.autocommit = True  # Включаем автокоммит для оптимизации транзакций чтения
    try:
        yield conn  # Возвращаем объект соединения в блок with
    except Exception as e:
        raise ConnError(f"Postgres connection failed: {e}") from e  # Пробрасываем ошибку подключения
    finally:
        conn.close()  # Закрываем соединение для освобождения серверных ресурсов

In [ ]:
def _resolve_pk(conn, db: str, schema: str, table: str) -> List[str]:  # Функция автоматического поиска первичного ключа (поддержка составного PK)
    cur = conn.cursor()  # Создаём курсор для выполнения мета-запроса
    try:  # Защищаем блок от синтаксических ошибок и проблем с правами
        if db == "ora":  # Ветка для Oracle
            sql = "SELECT cols.column_name, cols.position FROM all_constraints c JOIN all_cons_columns cols ON c.constraint_name = cols.constraint_name WHERE c.constraint_type = 'P' AND c.owner = :1 AND c.table_name = :2 ORDER BY cols.position"  # Формируем запрос к словарю данных Oracle с позицией колонки
            cur.execute(sql, (schema.upper(), table.upper()))  # Выполняем параметризированный запрос
        else:  # Ветка для PostgresPro
            sql = "SELECT kcu.column_name, kcu.ordinal_position FROM information_schema.table_constraints tc JOIN information_schema.key_column_usage kcu ON tc.constraint_name = kcu.constraint_name WHERE tc.constraint_type = 'PRIMARY KEY' AND tc.table_schema = %s AND tc.table_name = %s ORDER BY kcu.ordinal_position"  # Формируем запрос к стандартному каталогу SQL с позицией колонки
            cur.execute(sql, (schema, table))  # Выполняем запрос с параметрами
        res = cur.fetchall()  # Извлекаем все строки результата (поддержка составного PK)
        if not res: raise SchemaError(f"Primary key not found in {db}.{schema}.{table}. Provide key_col explicitly.")  # Бросаем ошибку если PK отсутствует
        return [r[0] for r in res]  # Возвращаем список имён колонок первичного ключа
    finally: cur.close()  # Освобождаем ресурсы курсора независимо от результата


def _validate_schema_compatibility(meta_ora: List[tuple], meta_pg: List[tuple], table_name: str) -> Dict[str, Any]:
    """Валидация совместимости схем перед сравнением. Возвращает отчёт о различиях."""
    issues = {"missing_columns": [], "type_mismatches": [], "nullable_mismatch": [], "compatible": True}
    ora_cols = {r[0].upper(): r for r in meta_ora}
    pg_cols = {r[0].lower(): r for r in meta_pg}
    
    # Проверка наличия колонок в обеих схемах
    for col_name in ora_cols:
        if col_name.lower() not in pg_cols:
            issues["missing_columns"].append(f"{col_name} missing in Postgres")
            issues["compatible"] = False
    for col_name in pg_cols:
        if col_name.upper() not in ora_cols:
            issues["missing_columns"].append(f"{col_name} missing in Oracle")
            issues["compatible"] = False
    
    # Проверка типов данных и nullable
    for col_name, ora_row in ora_cols.items():
        if col_name.lower() in pg_cols:
            pg_row = pg_cols[col_name.lower()]
            ora_type, ora_nullable = ora_row[1], ora_row[2] if len(ora_row) > 2 else None
            pg_type, pg_nullable = pg_row[1], pg_row[2] if len(pg_row) > 2 else None
            
            # Упрощённая проверка совместимости типов
            type_map = {
                'VARCHAR': ['character varying', 'text', 'varchar'],
                'NUMBER': ['numeric', 'integer', 'bigint', 'decimal'],
                'DATE': ['timestamp', 'date', 'timestamp without time zone'],
                'CLOB': ['text', 'character varying'],
                'BLOB': ['bytea', 'blob']
            }
            type_compatible = False
            for base_type, pg_types in type_map.items():
                if ora_type.startswith(base_type) and pg_type.lower() in pg_types:
                    type_compatible = True
                    break
            if not type_compatible and ora_type.split('(')[0] != pg_type.split('(')[0]:
                issues["type_mismatches"].append(f"{col_name}: {ora_type} vs {pg_type}")
                issues["compatible"] = False
            
            # Проверка NOT NULL совместимости
            if ora_nullable == 'N' and pg_nullable == 'YES':
                issues["nullable_mismatch"].append(f"{col_name}: NOT NULL in Oracle but nullable in Postgres")
    
    return issues

def _fetch_meta(conn, db: str, schema: str, table: str) -> List[tuple]:  # Функция сбора структурной информации таблицы
    cur = conn.cursor()  # Инициализируем рабочий курсор
    try:  # Открываем защищённый блок выполнения
        if db == "ora": cur.execute("SELECT column_name, data_type, nullable, data_length FROM all_tab_columns WHERE owner = :1 AND table_name = :2 ORDER BY column_id", (schema.upper(), table.upper()))  # Запрос метаданных Oracle
        else: cur.execute("SELECT column_name, data_type, is_nullable, character_maximum_length FROM information_schema.columns WHERE table_schema = %s AND table_name = %s ORDER BY ordinal_position", (schema, table))  # Запрос метаданных Postgres
        return [r for r in cur.fetchall()]  # Возвращаем список кортежей описания колонок
    finally: cur.close()  # Закрываем курсор для предотвращения утечек памяти

In [ ]:
def _stream_chunks(conn, db: str, schema: str, table: str, key: List[str], chunk: int = 100000, filters: Optional[Dict] = None) -> Generator[_pd.DataFrame, None, None]:  # Генератор потоковой выгрузки с защитой от OOM (поддержка составного PK)
    cur = conn.cursor()  # Создаём рабочий курсор для итеративного чтения
    if db == "ora": cur.arraysize = chunk  # Настраиваем размер сетевого буфера Oracle
    else: cur.execute("SET statement_timeout = '60000'")  # Устанавливаем таймаут выполнения запросов для Postgres
    last_val = None  # Инициализируем переменную состояния курсорной пагинации
    where = "WHERE 1=1"  # Устанавливаем безопасную базовую заглушку предиката
    key_cond = ""  # Инициализируем key_cond пустой строкой
    params = []  # Создаём пустой список для bind-переменных
    if filters:  # Проверяем наличие пользовательских условий фильтрации
        for k, v in filters.items():  # Итерируемся по словарю фильтров
            if db == "pg": where += f" AND {k} = %s"  # Добавляем условие для PostgresPro
            else: where += f" AND {k} = :{len(params)+1}"  # Добавляем условие для Oracle
            params.append(v)  # Сохраняем значение для безопасной параметризации
    # Формируем ORDER BY для составного ключа
    order_by = ", ".join(key) if isinstance(key, list) else key
    try:  # Запускаем защищённый цикл выборки
        while True:  # Бесконечный цикл до полного исчерпания данных
            # Формируем условие курсора для составного ключа
            if last_val is not None:
                key_cols = ", ".join(key)
                if db == "ora":
                    key_values = ", ".join([f":lk{i}" for i in range(len(key))])
                else:  # pg
                    key_values = ", ".join(["%s"] * len(key))
                key_cond = f" AND ROW({key_cols}) > ROW({key_values})"
            final_where = where + key_cond  # Объединяем фильтры и пагинацию в единый предикат
            if db == "ora":  # Ветка генерации SQL для Oracle
                qry = f"SELECT * FROM {schema}.{table} {final_where} ORDER BY {order_by} FETCH NEXT :lim ROWS ONLY"  # Формируем запрос с курсорным лимитом
                exec_p = {**{f":{i+1}": p for i, p in enumerate(params)}, ":lim": chunk}  # Собираем единый словарь параметров
                if last_val is not None:
                    if isinstance(last_val, (list, tuple)):
                        for i, v in enumerate(last_val): exec_p[f":lk{i}"] = v  # Добавляем состояние составного курсора
                    else: exec_p[":lk"] = last_val  # Добавляем состояние простого курсора
                cur.execute(qry, exec_p)  # Выполняем параметризированный запрос
            else:  # Ветка генерации SQL для PostgresPro
                qry = f"SELECT * FROM {schema}.{table} {final_where} ORDER BY {order_by} LIMIT %s"  # Формируем запрос с LIMIT
                exec_p = params[:]  # Копируем базовые параметры
                if last_val is not None:
                    if isinstance(last_val, (list, tuple)): exec_p.extend(last_val)  # Добавляем значения составного ключа
                    else: exec_p.append(last_val)  # Добавляем значение простого ключа
                exec_p.append(chunk)  # Добавляем лимит
                cur.execute(qry, exec_p)  # Выполняем запрос
            batch = cur.fetchall()  # Извлекаем пакет строк из буфера курсора
            if not batch: break  # Прерываем цикл при получении пустого результата
            df = _pd.DataFrame(batch)  # Конвертируем сырые данные в DataFrame
            # Сохраняем последнее значение ключа (составного или простого)
            first_key = key[0] if isinstance(key, list) else key
            if isinstance(key, list) and len(key) > 1:
                last_val = tuple(df[k].iloc[-1] for k in key)  # Кортеж значений для составного ключа
            else:
                last_val = df[first_key].iloc[-1]  # Значение для простого ключа
            yield df  # Возвращаем порцию данных вызывающему коду
            del batch, df  # Явно удаляем ссылки для освобождения памяти GC
    finally: cur.close()  # Гарантируем закрытие курсора при любом исходе

In [ ]:
class CompStrategy:
    def __init__(self, tol: float = 1e-6, case: bool = True, use_hash_for_large: bool = False, hash_threshold: int = 10_000_000):
        self.tol, self.case = tol, case
        self.use_hash_for_large = use_hash_for_large
        self.hash_threshold = hash_threshold
    
    def _compute_row_hash(self, df: '_pd.DataFrame', key_col: List[str]) -> '_pd.Series':
        cols_to_hash = [c for c in df.columns if c not in key_col]
        def hash_row(row):
            normalized = tuple('<<NULL>>' if _pd.isna(v) or v is None else v for v in row[cols_to_hash])
            return hashlib_md5(str(normalized).encode()).hexdigest()
        return df.apply(hash_row, axis=1)

    def _compare_columns_parallel(self, d1: '_pd.DataFrame', d2: '_pd.DataFrame', common: List[str], max_workers: int = 4) -> Dict[str, int]:
        diffs = {}
        def compare_column(col: str) -> tuple:
            v1, v2 = d1[col].values, d2[col].values
            total = max(len(d1), len(d2))
            if np_issubdtype(v1.dtype, np_number) and np_issubdtype(v2.dtype, np_number):
                v1_float = v1.astype(float)
                v2_float = v2.astype(float)
                mask = ~np_isclose(v1_float, v2_float, atol=self.tol, equal_nan=False, rtol=0)
            else:
                def normalize_val(v):
                    if v is None or (np_isscalar(v) and np_isnan(v)):
                        return '<<NULL>>'
                    return str(v)
                v1_str = np_array([normalize_val(x) for x in v1])
                v2_str = np_array([normalize_val(x) for x in v2])
                if self.case:
                    mask = v1_str != v2_str
                else:
                    mask = np_char_lower(v1_str) != np_char_lower(v2_str)
            return col, int(np_count_nonzero(mask[:total]))
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(compare_column, col): col for col in common}
            for future in as_completed(futures):
                col, diff_count = future.result()
                diffs[col] = diff_count
        return diffs

    def run(self, d1: '_pd.DataFrame', d2: '_pd.DataFrame', key: List[str], cache: Dict, total_rows_checked: int = 0) -> Dict[str, Any]:
        if d1.empty or d2.empty:
            return {"mismatches": 0, "diffs": {}, "cnt": 0}
        if len(d1) != len(d2):
            logger_warning(f"Chunk length mismatch: Oracle={len(d1)}, Postgres={len(d2)}. Aligning by key...")
        key_list = key if isinstance(key, list) else [key]
        try:
            merged = d1.merge(d2, on=key_list, how='outer', suffixes=('_ora', '_pg'), indicator=True)
            matched = merged[merged['_merge'] == 'both'].reset_index(drop=True)
            if matched.empty:
                total = max(len(d1), len(d2))
                common_cols = d1.columns.intersection(d2.columns).tolist()
                return {"total": total, "diffs": {c: total for c in common_cols}}
            d1_matched = matched[[c + '_ora' if c not in key_list else c for c in d1.columns]].copy()
            d2_matched = matched[[c + '_pg' if c not in key_list else c for c in d2.columns]].copy()
            d1_matched.columns = list(d1.columns)
            d2_matched.columns = list(d2.columns)
            unmatched_count = len(merged[merged['_merge'] != 'both'])
        except Exception as e:
            logger_warning(f"Merge failed ({e}), falling back to sort-based comparison")
            d1_matched = d1.sort_values(by=key_list).reset_index(drop=True)
            d2_matched = d2.sort_values(by=key_list).reset_index(drop=True)
            unmatched_count = abs(len(d1) - len(d2))
        common = d1_matched.columns.intersection(d2_matched.columns).tolist()
        total = len(d1_matched) + unmatched_count
        if self.use_hash_for_large and total_rows_checked + total > self.hash_threshold:
            logger_info(f"Switching to hash validation at row {total_rows_checked + total}")
            hash1 = self._compute_row_hash(d1_matched, key_list)
            hash2 = self._compute_row_hash(d2_matched, key_list)
            mismatches = int((hash1 != hash2).sum()) + unmatched_count
            diffs = {c: mismatches for c in common}
        else:
            diffs = self._compare_columns_parallel(d1_matched, d2_matched, common, max_workers=4)
            if unmatched_count > 0:
                for c in diffs:
                    diffs[c] = diffs.get(c, 0) + unmatched_count
        for c in common[:5]:
            stats = d1_matched[c].describe()
            if np_issubdtype(d1_matched[c].dtype, np_number):
                cache[c] = {"mean": float(stats.get("mean", 0)), "std": float(stats.get("std", 0)), "min": float(stats.get("min", 0)), "max": float(stats.get("max", 0))}
            else:
                cache[f"{c}_top"] = dict(d1_matched[c].value_counts().head(3))
        return {"total": total, "diffs": diffs}

In [ ]:
def _draw_dashboard(cnts: Dict, matrix: '_pd.DataFrame', stats: Dict):
    from matplotlib.table import table as mpl_table
    _plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "figure.figsize": (14, 10)})
    fig, ax = _plt.subplots(2, 2, constrained_layout=True)
    _sns.barplot(x=["Oracle", "PostgresPro"], y=[cnts.get("ora", 0), cnts.get("pg", 0)], ax=ax[0,0], palette="muted")
    ax[0,0].set_title("Row Count")
    if not matrix.empty:
        _sns.heatmap(matrix, annot=True, cmap="RdYlGn", fmt=".1f", ax=ax[0,1], vmin=0, vmax=100)
    ax[0,1].set_title("Match %")
    num_cols = [k for k in stats if "top" not in k and isinstance(stats.get(k), dict)]
    if num_cols and num_cols[0] in stats:
        col_name = num_cols[0]
        col_data = stats[col_name]
        if isinstance(col_data, dict) and "mean" in col_data:
            mean_val = col_data.get("mean", 0)
            std_val = max(col_data.get("std", 1), 0.001)
            sample_data = _np.random.normal(mean_val, std_val, 1000)
            _sns.kdeplot(data=sample_data, ax=ax[1,0])
            ax[1,0].set_title(f"Dist: {col_name}")
        else:
            ax[1,0].text(0.5, 0.5, "Invalid stats format", ha="center")
    else:
        ax[1,0].text(0.5, 0.5, "No numeric", ha="center")
    ax[1,1].axis("off")
    tbl_data = [[k, str(v)[:50]+"..." if len(str(v))>50 else str(v)] for k,v in list(stats.items())[:8]]
    mpl_table(ax[1,1], cellText=tbl_data, colLabels=["Key", "Val"], loc="center", cellLoc="left")
    _plt.suptitle("Oracle vs PostgresPro")
    _plt.show()

In [ ]:
def compare(schema_ora: str, schema_pg: str, tables: List[str], cfg_ora: Dict, cfg_pg: Dict, key_col: Optional[str] = None, filters: Optional[Dict] = None, max_rows: int = 5_000_000, chunk: int = 100000, use_hash_for_large: bool = False, validate_schema: bool = True) -> Dict[str, Any]:
    from itertools import zip_longest
    assert len(tables) == 2, "Требуется ровно 2 таблицы для попарного сравнения"
    strat = CompStrategy(use_hash_for_large=use_hash_for_large, hash_threshold=10_000_000)
    cnts, diffs, cache, checked = {"ora": 0, "pg": 0}, {}, {}, 0
    start_time = _time.time()

    with _ora_conn(cfg_ora) as c_ora, _pg_conn(cfg_pg) as c_pg:
        _logger.info("Resolving keys & metadata...")
        meta_ora = _fetch_meta(c_ora, "ora", schema_ora, tables[0])
        meta_pg = _fetch_meta(c_pg, "pg", schema_pg, tables[1])
        ora_columns = {r[0].upper() for r in meta_ora}
        pg_columns = {r[0].lower() for r in meta_pg}
        k_ora = key_col or _resolve_pk(c_ora, "ora", schema_ora, tables[0])
        k_pg = key_col or _resolve_pk(c_pg, "pg", schema_pg, tables[1])
        k_ora_list = k_ora if isinstance(k_ora, list) else [k_ora]
        k_pg_list = k_pg if isinstance(k_pg, list) else [k_pg]
        for k in k_ora_list:
            if k.upper() not in ora_columns:
                raise SchemaError(f"Key column '{k}' not found in Oracle table {schema_ora}.{tables[0]}. Available columns: {sorted(ora_columns)}")
        for k in k_pg_list:
            if k.lower() not in pg_columns:
                raise SchemaError(f"Key column '{k}' not found in Postgres table {schema_pg}.{tables[1]}. Available columns: {sorted(pg_columns)}")
        if k_ora != k_pg:
            _logger.warning(f"Key mismatch: {k_ora} vs {k_pg}. Ensure semantic equivalence.")
        if validate_schema:
            _logger.info("Validating schema compatibility...")
            schema_issues = _validate_schema_compatibility(meta_ora, meta_pg, f"{tables[0]} vs {tables[1]}")
            if not schema_issues["compatible"]:
                _logger.error(f"Schema validation failed: {schema_issues}")
                raise SchemaError(f"Schema incompatibility detected: missing={schema_issues['missing_columns']}, type_mismatches={schema_issues['type_mismatches']}")
            _logger.info("Schema validation passed")
        _logger.info(f"Using key columns: Oracle={k_ora_list}, Postgres={k_pg_list}")
        it1 = _stream_chunks(c_ora, "ora", schema_ora, tables[0], k_ora_list, chunk, filters)
        it2 = _stream_chunks(c_pg, "pg", schema_pg, tables[1], k_pg_list, chunk, filters)
        total_est = max_rows // chunk
        pbar = _tqdm(desc="Comparing chunks", total=total_est, unit="chunk")
        for d1, d2 in zip_longest(it1, it2, fillvalue=None):
            chunk_start = _time.time()
            if d1 is None or d2 is None:
                if d1 is None and d2 is not None:
                    _logger.warning(f"Oracle iterator exhausted, but Postgres still has {len(d2)} rows")
                    cnts["pg"] += len(d2)
                elif d2 is None and d1 is not None:
                    _logger.warning(f"Postgres iterator exhausted, but Oracle still has {len(d1)} rows")
                    cnts["ora"] += len(d1)
                break
            res = strat.run(d1, d2, k_ora_list, cache, checked)
            chunk_time = _time.time() - chunk_start
            cnts["ora"], cnts["pg"] = cnts["ora"] + len(d1), cnts["pg"] + len(d2)
            checked += res["total"]
            for c, n in res["diffs"].items():
                diffs[c] = diffs.get(c, 0) + n
            pbar.set_postfix({"rows": checked, "eta": f"{_time.strftime('%H:%M:%S', _time.gmtime(chunk_time * (total_est - pbar.n)))}"})
            pbar.update(1)
            if checked >= max_rows:
                _logger.info(f"Limit {max_rows} reached. Stopping early.")
                break
            d1 = None
            d2 = None
            res = None
        pbar.close()
        elapsed = _time.time() - start_time
        _logger.info(f"Comparison completed in {elapsed:.2f}s. Rows/sec: {checked/elapsed:.0f}")
        match = {k: round((1 - v/checked)*100, 2) if checked else 100.0 for k, v in diffs.items()}
        _draw_dashboard(cnts, _pd.DataFrame([match]), cache)
    return {"counts": cnts, "match_pct": match, "stats": cache, "elapsed_seconds": elapsed, "rows_per_second": checked/elapsed if elapsed > 0 else 0}

In [ ]:
# 📦 КОНФИГУРАЦИЯ И ЗАПУСК
# 💡 Совет: используйте переменные окружения для конфиденциальных данных
ORA_CFG = {
    "user": _os.environ.get("ORA_USER", "app_user"),
    "password": _os.environ.get("ORA_PASSWORD", "secure_pwd"),
    "dsn": _os.environ.get("ORA_DSN", "ora_host:1521/ORCL")
}
PG_CFG = {
    "host": _os.environ.get("PG_HOST", "pg_host"),
    "port": int(_os.environ.get("PG_PORT", 5432)),
    "dbname": _os.environ.get("PG_DBNAME", "analytics"),
    "user": _os.environ.get("PG_USER", "pg_user"),
    "password": _os.environ.get("PG_PASSWORD", "secure_pwd")
}

# ✅ Вариант 1: Сравнение с валидацией схем и параллельной обработкой
report_validated = compare(
    schema_ora="LEGACY_SRC", schema_pg="public",
    tables=["orders", "orders_replica"],
    cfg_ora=ORA_CFG, cfg_pg=PG_CFG,
    filters={"status": "active", "region": "EU"},
    max_rows=5_000_000,
    chunk=100000,
    validate_schema=True,  # Включить валидацию совместимости схем
    use_hash_for_large=False  # Использовать хэш-валидацию для >10M строк
)

# ✅ Вариант 2: Хэш-валидация для очень больших таблиц (>10M строк)
report_hash = compare(
    schema_ora="LEGACY_SRC", schema_pg="target_db",
    tables=["customers", "clients"],
    cfg_ora=ORA_CFG, cfg_pg=PG_CFG,
    filters=None,
    max_rows=15_000_000,
    chunk=200000,  # Увеличенный размер чанка для производительности
    validate_schema=True,
    use_hash_for_large=True  # Автоматическое переключение на хэш-валидацию после 10M строк
)

# ✅ Вариант 3: Быстрое сравнение без валидации схем (если схемы гарантированно идентичны)
report_fast = compare(
    schema_ora="SRC", schema_pg="TGT",
    tables=["table1", "table1"],
    cfg_ora=ORA_CFG, cfg_pg=PG_CFG,
    validate_schema=False,  # Пропустить валидацию для ускорения
    max_rows=1_000_000
)

# 💡 Результаты содержат метрики производительности:
# report["elapsed_seconds"] - общее время выполнения
# report["rows_per_second"] - скорость обработки
# report["counts"] - количество строк в источнике и цели
# report["match_pct"] - процент совпадений по колонкам
# report["stats"] - статистика данных


## 📊 Примечания к эксплуатации

### 🔧 Новые возможности
| Функция | Описание | Когда использовать |
|---------|----------|-------------------|
| **Валидация схем** | `_validate_schema_compatibility()` проверяет совместимость типов, наличие колонок и NOT NULL ограничений | Перед первым сравнением новых таблиц |
| **Параллельное сравнение** | ThreadPoolExecutor для одновременной обработки колонок (4 потока) | Таблицы с >20 колонками |
| **Хэш-валидация** | MD5 хэширование строк для таблиц >10M строк | Очень большие объёмы данных |
| **Прогресс с ETA** | Оценка времени до завершения в реальном времени | Длительные операции сравнения |
| **Логирование в файл** | Запись логов в `comparator.log` (настраивается через `COMPARATOR_LOG_FILE`) | Аудит и отладка |
| **Конфигурация через ENV** | Поддержка переменных окружения для учётных данных | CI/CD и production |

### ⚡ Производительность
| Размер таблицы | Рекомендуемый chunk | Режим | Ожидаемая скорость |
|---------------|---------------------|-------|--------------------|
| <1M строк | 50,000 | Стандартный | ~50K строк/сек |
| 1-10M строк | 100,000 | Параллельный | ~200K строк/сек |
| >10M строк | 200,000 | Хэш-валидация | ~500K строк/сек |

### 🛡️ Валидация схем
Функция `_validate_schema_compatibility()` выполняет:
- Проверку наличия всех колонок в обеих схемах
- Сопоставление типов данных (VARCHAR↔text, NUMBER↔numeric, DATE↔timestamp)
- Проверку NOT NULL ограничений
- Возвращает детальный отчёт о несовместимостях

### 📁 Переменные окружения
```bash
export ORA_USER="app_user"
export ORA_PASSWORD="secure_pwd"
export ORA_DSN="ora_host:1521/ORCL"
export PG_HOST="pg_host"
export PG_PORT=5432
export PG_DBNAME="analytics"
export PG_USER="pg_user"
export PG_PASSWORD="secure_pwd"
export COMPARATOR_LOG_FILE="/var/log/comparator.log"
```

💡 **Запуск:** Для таблиц >10M строк используйте `use_hash_for_large=True` и убедитесь, что ключевая колонка имеет B-Tree индекс в обеих БД.
